# Topic Model

## Installation

In [ ]:
# !pip install bertopic

## Loading Dataset

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
import pandas as pd

df = pd.read_csv('food.csv')
df.head()

In [ ]:
docs = df['介紹'].tolist()
docs[:2]

## Getting embeddings and training the model

There are multiple models we can use to turn the documents into embeddings:
```
paraphrase-multilingual-MiniLM-L12-v2
paraphrase-multilingual-mpnet-base-v2
distiluse-base-multilingual-cased-v2
xlm-r-bert-base-nli-stsb-mean-tokens
```

In [ ]:
# Get embeddings
sentence_model = SentenceTransformer("distiluse-base-multilingual-cased-v2") # 
embeddings = sentence_model.encode(docs, show_progress_bar=True)

In [ ]:
topic_model = BERTopic(language="multilingual",
                       min_topic_size=2, # At least 20 documents per topic
                       # nr_topics=10, # This will specify topic reduction
                       verbose=True)
topics, probs = topic_model.fit_transform(docs, embeddings)

The training process will yield two outputs:
1. `topics` gives you the topic number of each document, as a list
2. `probs` gives you the probability of the document belonging to the topic (higher number means the model is very confident)

In [ ]:
# The easiest way to combine the result back to the dataset is to...
df['topic'] = topics

In [ ]:
df.head()

## Interpreting the model

In [ ]:
topic_model.get_topic_info()

In [ ]:
topic_model.get_topic(0)

In [ ]:
topic_model.get_document_info(docs)

In [ ]:
# # This is how you save and load a topic model
# topic_model.save("my_model", serialization="pickle")
# loaded_model = BERTopic.load("MaartenGr/BERTopic_Wikipedia")

## Visualizations

The first task of interpreting a topic model is often assigning a label to each topic.

In [ ]:
topic_model.visualize_barchart()

In [ ]:
fig_topics = topic_model.visualize_topics()
fig_topics

In [ ]:
topic_model.visualize_heatmap()

In [ ]:
from umap import UMAP
reduced_embeddings = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine').fit_transform(embeddings)
topic_model.visualize_documents(docs, reduced_embeddings=reduced_embeddings)

## Validation

Topic modelling is exploratory, so validation is a design requirement rather than an afterthought. 

The simplest way is:
1. sample a subset of documents
2. ask coder(s) to annotate them based on topic label
3. calculate intercoder reliability between the results from the topic model and the manual annotation

In [ ]:
# Making a sample dataset for manual validation.

doc_df = topic_model.get_document_info(docs)

sample_df = doc_df.groupby('Topic').sample(n=3)
sample_df = sample_df.sample(frac=1)
sample_df = sample_df[sample_df.Topic != -1]
sample_df = sample_df[['Document','Topic']]
sample_df

In [ ]:
sample_df.to_csv('sample.csv')

## Some last words

Check this out for more functions: https://maartengr.github.io/BERTopic/